<a href="https://colab.research.google.com/github/tobiartinian/Econom-a-y-Finanzas/blob/main/Submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.0 MB/s eta 0:00:00


In [4]:
import lightgbm as lgb
import duckdb
import optuna
import pandas as pd
import os

# 1. Conectar a DuckDB
con = duckdb.connect("competencia.duckdb")

# 2. Verificar si la tabla competencia_01 existe, y si no, recrearla desde el CSV
tablas_existentes = [row[0] for row in con.execute("SHOW TABLES;").fetchall()]

if "competencia_01" not in tablas_existentes:
    print("⚠️ La tabla 'competencia_01' no está en esta sesión. Generándola desde el CSV...")

    file_path = "/content/drive/MyDrive/Economía y finanzas/competencia_01.csv" # Ajusta tu ruta si es necesario

    # Leer header para las variables con lag
    df_header = pd.read_csv(file_path, nrows=0)
    excluir = {'foto_mes', 'numero_de_cliente', 'clase_ternaria'}
    variables_lag_validas = [col for col in df_header.columns if col not in excluir]

    lags_sql = ",\n    ".join([
        f"LAG({col}, 1) OVER (PARTITION BY numero_de_cliente ORDER BY foto_mes) AS {col}_lag1"
        for col in variables_lag_validas
    ])

    query_create = f"""
    CREATE OR REPLACE TABLE competencia_01 AS
    SELECT
        *,
        {lags_sql}
    FROM read_csv_auto('{file_path}');
    """
    con.execute(query_create)
    print("¡Tabla creada y cargada con éxito en DuckDB!")
else:
    print("✅ La tabla 'competencia_01' ya existe en DuckDB.")

# 3. Recuperar los mejores hiperparámetros de Optuna
storage_name = "sqlite:////content/drive/MyDrive/DMEyF/2026/notebooks/data/optimization_lgbm.db"
study_name = "exp_101_lgbm-opt"

study = optuna.load_study(study_name=study_name, storage=storage_name)
best_params = study.best_params

print("\n🏆 Mejores hiperparámetros recuperados de Optuna:")
for k, v in best_params.items():
    print(f"  - {k}: {v}")

best_params.update({
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "random_state": 772859,
    "n_jobs": 1,
    "verbose": -1
})

# 4. Extraer features y entrenar
df_header = con.execute("SELECT * FROM competencia_01 LIMIT 0").df()
excluir = {"foto_mes", "numero_de_cliente", "clase_ternaria"}
features = [col for col in df_header.columns if col not in excluir]
features_sql = ", ".join(features)

print("\nCargando datos de entrenamiento desde DuckDB...")
query_train = f"""
    SELECT
        {features_sql},
        CASE WHEN clase_ternaria = 'BAJA+2' THEN 1 ELSE 0 END AS target
    FROM competencia_01
    WHERE foto_mes BETWEEN 202103 AND 202106
"""
df_train = con.execute(query_train).df()

X_train = df_train.drop(columns=["target"])
y_train = df_train["target"]

print("Entrenando el modelo final de LightGBM...")
train_data = lgb.Dataset(X_train, label=y_train)

model = lgb.train(
    best_params,
    train_data,
    num_boost_round=400
)

# 5. Cargar datos de test y generar submission
mes_test = 202108
print(f"Cargando datos del mes de test ({mes_test})...")

query_test = f"""
    SELECT *
    FROM competencia_01
    WHERE foto_mes = {mes_test}
"""
df_test = con.execute(query_test).df()
X_test = df_test[features]

print("Generando predicciones...")
df_test['probabilidad'] = model.predict(X_test)

umbral_optimo = 0.025
df_bajas_predichas = df_test[df_test['probabilidad'] >= umbral_optimo].copy()
df_submission = df_bajas_predichas[['numero_de_cliente']].copy()

output_filename = 'submission_final.csv'
df_submission.to_csv(output_filename, index=False, header=False)

print(f"\n✨ ¡Listo! Se filtraron y guardaron {len(df_submission)} clientes predichos como baja+2 en '{output_filename}'.")
df_submission.head(10)

⚠️ La tabla 'competencia_01' no está en esta sesión. Generándola desde el CSV...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

¡Tabla creada y cargada con éxito en DuckDB!

🏆 Mejores hiperparámetros recuperados de Optuna:
  - learning_rate: 0.05407060814866634
  - num_leaves: 20
  - min_child_samples: 355
  - feature_fraction: 0.7683542504735678
  - bagging_fraction: 0.7923122547071753
  - bagging_freq: 1

Cargando datos de entrenamiento desde DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Entrenando el modelo final de LightGBM...
Cargando datos del mes de test (202108)...
Generando predicciones...

✨ ¡Listo! Se filtraron y guardaron 8295 clientes predichos como baja+2 en 'submission_final.csv'.


,numero_de_cliente
8,70261782
25,70398446
36,70509939
41,70683182
46,70696276
52,70730197
61,70851465
64,70864794
65,70869523
90,71002248
